# Task 3: Caching Experiment for all queries

- Caches a reduced `integrated_taxi_trips` view containing only the columns used by the queries.
- Compares Queries before and after caching.

## Connect to Spark and imports

In [1]:
import sys
from time import perf_counter

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.runner import (
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)
from analytics.benchmarking import measure_registered_query, rows_match

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-Caching-Queries3And5')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 12:03:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Register the uncached Gold table

In [2]:
# Prepare the weather lookup before running any timed queries.
from delta.tables import DeltaTable
from weather_conditions_job import load_coco_codes

weather_lookup_path = 's3a://delta-data/silver/silver_weather_conditions'
if not DeltaTable.isDeltaTable(spark, weather_lookup_path):
    load_coco_codes(spark)

spark.catalog.clearCache()
register_integrated_taxi_trips(spark)
spark.read.format("delta") \
    .load("s3a://delta-data/silver/silver_weather_conditions") \
    .createOrReplaceTempView("weather_conditions")

assert not spark.catalog.isCached(INTEGRATED_TABLE_VIEW)
spark.table(INTEGRATED_TABLE_VIEW).printSchema()

26/09/20 12:03:46 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- pickup_date: string (nullable = true)
 |-- dropoff_date: string (nullable = true)
 |-- trip_id: long (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- dropoff_zone_name: string (nullable = true)
 |-- dropof

26/09/20 12:03:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Measure uncached execution

In [3]:
QUERY_NAMES = (
    'query_03_air_quality_vs_demand',
    'query_05_peak_travel_hours',
    'query_01_monthly_zone_demand',
    'query_06_monthly_taxi_demand_trend',
    'query_04',
    'query_02',
)
REPETITIONS = 9

uncached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

print('Uncached measurements complete.')

Uncached measurements complete.


## Uncached physical plan

In [4]:
execute_registered_query(
    spark,
    'query_03_air_quality_vs_demand',
).explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (20)
+- Sort (19)
   +- Exchange (18)
      +- BroadcastNestedLoopJoin Cross BuildRight (17)
         :- HashAggregate (6)
         :  +- Exchange (5)
         :     +- HashAggregate (4)
         :        +- Project (3)
         :           +- Filter (2)
         :              +- Scan parquet  (1)
         +- BroadcastExchange (16)
            +- HashAggregate (15)
               +- Exchange (14)
                  +- HashAggregate (13)
                     +- HashAggregate (12)
                        +- Exchange (11)
                           +- HashAggregate (10)
                              +- Project (9)
                                 +- Filter (8)
                                    +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [pu_datetime_hour#12754, average_value_pu#12771, date#12785]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(average_value_pu)]
Rea

## Cache gold table

In [5]:
CACHE_COLUMNS = (
    'pickup_datetime',
    'pu_datetime_hour',
    'average_value_pu',
    'pickup_date',
    'pickup_zone_name',
    'coco_pu',
    'trip_distance'
)

cache_input = spark.table(INTEGRATED_TABLE_VIEW).select(*CACHE_COLUMNS)
cache_input.createOrReplaceTempView(INTEGRATED_TABLE_VIEW)

cache2 = spark.read.format("delta") \
    .load("s3a://delta-data/silver/silver_weather_conditions") \
    .createOrReplaceTempView("weather_conditions")



cache_started_at = perf_counter()
spark.catalog.cacheTable(INTEGRATED_TABLE_VIEW)
spark.catalog.cacheTable('weather_conditions')
cached_input_rows = spark.table('weather_conditions').count()
cached_input_rows = spark.table(INTEGRATED_TABLE_VIEW).count()
cache_materialization_seconds = perf_counter() - cache_started_at

assert spark.catalog.isCached(INTEGRATED_TABLE_VIEW)

print(f'Cached input rows: {cached_input_rows}')
print(f'Cache materialization: {cache_materialization_seconds:.3f} seconds')

Cached input rows: 9417833
Cache materialization: 5.071 seconds


## Measure cached execution and verify results

In [6]:
cached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

for query_name in QUERY_NAMES:
    assert rows_match(
        uncached_results[query_name]['rows'],
        cached_results[query_name]['rows'],
    ), (
        f'Results differ for {query_name}'
    )

print('All cached results match the uncached results within float tolerance.')

All cached results match the uncached results within float tolerance.


## Cached physical plan

- `Scan In-memory table` confirms that Spark uses the cache.

In [7]:
execute_registered_query(
    spark,
    'query_03_air_quality_vs_demand',
).explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (23)
+- Sort (22)
   +- Exchange (21)
      +- BroadcastNestedLoopJoin Cross BuildRight (20)
         :- HashAggregate (9)
         :  +- Exchange (8)
         :     +- HashAggregate (7)
         :        +- Filter (6)
         :           +- Scan In-memory table integrated_taxi_trips (1)
         :                 +- InMemoryRelation (2)
         :                       +- * Project (5)
         :                          +- * ColumnarToRow (4)
         :                             +- Scan parquet  (3)
         +- BroadcastExchange (19)
            +- HashAggregate (18)
               +- Exchange (17)
                  +- HashAggregate (16)
                     +- HashAggregate (15)
                        +- Exchange (14)
                           +- HashAggregate (13)
                              +- Filter (12)
                                 +- Scan In-memory table integrated_taxi_trips (10)
                                       +- InMemor

## Benchmark summary

In [8]:
summary_rows = []
for query_name in QUERY_NAMES:
    uncached_median = uncached_results[query_name]['median_seconds']
    cached_median = cached_results[query_name]['median_seconds']
    summary_rows.append({
        'query': query_name,
        'uncached_samples_seconds': [
            round(value, 3)
            for value in uncached_results[query_name]['samples_seconds']
        ],
        'uncached_median_seconds': round(uncached_median, 3),
        'cached_samples_seconds': [
            round(value, 3)
            for value in cached_results[query_name]['samples_seconds']
        ],
        'cached_median_seconds': round(cached_median, 3),
        'speedup_factor': round(uncached_median / cached_median, 2),
        'cache_materialization_seconds': round(cache_materialization_seconds, 3),
    })

summary_rows

[{'query': 'query_03_air_quality_vs_demand',
  'uncached_samples_seconds': [4.254,
   3.155,
   2.719,
   1.898,
   1.845,
   1.77,
   2.0,
   2.288,
   1.793],
  'uncached_median_seconds': 2.0,
  'cached_samples_seconds': [0.575,
   0.643,
   0.459,
   0.68,
   0.59,
   0.616,
   0.531,
   0.601,
   0.811],
  'cached_median_seconds': 0.601,
  'speedup_factor': 3.33,
  'cache_materialization_seconds': 5.071},
 {'query': 'query_05_peak_travel_hours',
  'uncached_samples_seconds': [1.996,
   1.96,
   1.732,
   1.706,
   1.662,
   2.097,
   1.824,
   1.97,
   2.642],
  'uncached_median_seconds': 1.96,
  'cached_samples_seconds': [1.108,
   0.917,
   0.99,
   0.827,
   0.825,
   0.812,
   0.741,
   0.753,
   0.751],
  'cached_median_seconds': 0.825,
  'speedup_factor': 2.37,
  'cache_materialization_seconds': 5.071},
 {'query': 'query_01_monthly_zone_demand',
  'uncached_samples_seconds': [1.518,
   2.659,
   1.725,
   1.09,
   1.53,
   1.803,
   1.258,
   1.465,
   1.397],
  'uncached_med

## Advanced Caching: Shared Monthly Aggregation for Queries 1 & 6

- In this section, we test a more aggressive caching strategy for queries 1 & 6.
- Instead of just caching the raw columns, we cache the result of the aggregation by month and zone.
- This result is then used as a shared intermediate step for both Query 1 and Query 6, eliminating redundant computation across both queries.

In [9]:
print("\nMeasuring Shared Monthly Aggregation Caching...")
spark.catalog.clearCache()
register_integrated_taxi_trips(spark)

# 1. Create the intermediate shared aggregation
shared_agg_view = "shared_monthly_zone_agg"
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW {shared_agg_view} AS
    SELECT
        substring(pickup_date, 1, 7) as pickup_month,
        pickup_zone_name,
        count(*) as demand
    FROM integrated_taxi_trips
    GROUP BY 1, 2
""")

# 2. Cache and Materialize
start_mat = perf_counter()
spark.catalog.cacheTable(shared_agg_view)
spark.table(shared_agg_view).count()
mat_time = perf_counter() - start_mat
print(f"Shared Agg Materialization Time: {mat_time:.3f}s")

# 3. Measure Shared Cache Performance using repetitions for consistency
shared_results = {}
for name in ["query_01_monthly_zone_demand", "query_06_monthly_taxi_demand_trend"]:
    samples = []
    for _ in range(REPETITIONS):
        if name == 'query_01_monthly_zone_demand':
            start = perf_counter()
            spark.table(shared_agg_view).orderBy("pickup_zone_name", "pickup_month").collect()
        else: # query_06
            start = perf_counter()
            spark.table(shared_agg_view).groupBy("pickup_month").sum("demand").orderBy("pickup_month").collect()
        samples.append(perf_counter() - start)

    import statistics
    shared_results[name] = {
        'median_seconds': statistics.median(samples),
        'samples_seconds': samples
    }

print("Shared Aggregation measurements complete.")


Measuring Shared Monthly Aggregation Caching...


Shared Agg Materialization Time: 1.660s
Shared Aggregation measurements complete.


## Benchmark Summary for 1 & 6

In [10]:
print(f"{'Query':<35} | {'Baseline':<12} | {'Col-Cache':<12} | {'Co-Cache':<12} | {'Improvement (Col vs Co)':<20}")
print("-" * 95)

for q in ["query_01_monthly_zone_demand", "query_06_monthly_taxi_demand_trend"]:
    base = uncached_results[q]['median_seconds']
    col = cached_results[q]['median_seconds']
    co = shared_results[q]['median_seconds']
    diff = col - co

    print(f"{q:<35} | {base:>11.3f}s | {col:>11.3f}s | {co:>11.3f}s | {diff:>18.3f}s")

print("\nNote: 'Improvement' shows the time saved by using Shared Aggregation over Column-based caching.")

Query                               | Baseline     | Col-Cache    | Co-Cache     | Improvement (Col vs Co)
-----------------------------------------------------------------------------------------------
query_01_monthly_zone_demand        |       1.518s |       0.512s |       0.127s |              0.385s
query_06_monthly_taxi_demand_trend  |       0.954s |       0.326s |       0.128s |              0.198s

Note: 'Improvement' shows the time saved by using Shared Aggregation over Column-based caching.


## Cleanup

Run this cell only after saving the summary and plan output.

In [11]:
spark.catalog.uncacheTable(INTEGRATED_TABLE_VIEW)
spark.catalog.clearCache()
spark.stop()